### DPO 
### Direct Preference Optimization 


In [1]:
%pip install peft


/Users/jeldy/Documents/AUTODICDACT/AI:DL:ML:RAG:Agent/research_dojo/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch 
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import LoraConfig


In [6]:
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

#SFT
SFT_DATASET = "databricks/databricks-dolly-15k"
SFT_MAX_SAMPLES = 3000

#DPO
DPO_DATASET = "Anthropic/hh-rlhf"
DPO_MAX_SAMPLES = 5000 

# LORA 
LORA_R = 16 
LORA_ALPHA = 32
LORA_DROPOUT = 0.05 
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training 
MAX_SEQ_LEN = 512
MAX_PROMPT_LEN = 32

SFT_OUTPUT_DIR = "./checkpoints/sft"
SFT_EPOCHS = 1
SFT_BATCH_SIZE = 4
SFT_LR = 2e-4

DPO_OUTPUT_DIR = "./checkpoints/dpo"
DPO_EPOCHS = 1
DPO_BATCH_SIZE = 2               # DPO forwards chosen + rejected (+ implicit ref) each step, so keep this smaller than SFT_BATCH_SIZE
DPO_GRAD_ACCUM = 8
DPO_LR = 5e-6                    # DPO is far more sensitive to LR than SFT — the paper uses learning rates in this range, not 2e-4
DPO_BETA = 0.1 


In [7]:
import numpy as np 

dpo = load_dataset(DPO_DATASET)

Generating test split: 100%|██████████| 8552/8552 [00:00<00:00, 60485.15 examples/s]


In [ ]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context: 
        prompt = f"### Instruction:\n {instruction} \n\n ### Context:\n {context}\n\n### Response:\n"
    else: 
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {"text": prompt + response}

def train_sft():
    print(f"Loading base model: {BASE_MODEL}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, 
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        device_map = "auto", 
    )

    dataset = load_dataset(SFT_DATASET, split = "train")
    if SFT_MAX_SAMPLES:
        dataset = dataset.select(range(min(SFT_MAX_SAMPLES, len(dataset)))) 

    dataset = dataset.map(format_example)


    lora_config = LoraConfig(
        r = LORA_R,
        lora_alpha = LORA_ALPHA, 
        lora_dropout = LORA_DROPOUT, 
        target_modules = LORA_TARGET_MODULES,
        bias = "none",
        task_type = "CAUSAL_LM"
    )


    sft_args = SFTConfig(
        output_dir=SFT_OUTPUT_DIR,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=4,
        learning_rate=SFT_LR,
        logging_steps=10,
        save_strategy="epoch",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        max_length=MAX_SEQ_LEN,
        dataset_text_field="text",
        report_to="none",
    )

    sft_trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=dataset,
        peft_config=lora_config,
        processing_class=tokenizer,
    )

    sft_trainer.train()
    sft_trainer.save_model(SFT_OUTPUT_DIR)
    tokenizer.save_pretrained(SFT_OUTPUT_DIR)

    print(f"SFT model saved to {SFT_OUTPUT_DIR}")


train_sft()



Loading base model: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.936265


In [ ]:
def build_preference_dataset():
    pre_ds = load_dataset(DPO_DATASET, split = "train")
    if DPO_MAX_SAMPLES:
        pre_ds = pre_ds.select(range(min(DPO_MAX_SAMPLES, len(pre_ds))))
 
    # [User Prompt text here]\n\nAssistant: [AI Response text here]
    split_token = "\n\nAssistant:"


    def split_prompt_response(example):
        chosen_idx = example["chosen"].rfind(split_token) # find right to left, reverse find is used to locate the very last exchange
        rejected_idx = example["rejected"].rfind(split_token)

        if chosen_idx == -1 or rejected_idx == -1: 
            """If the reverse find does not locate any text in the chosen or rejected text, then it will return -1"""
            raise ValueError("Split token not found in the chosen or rejected text.")

        chosen_split = chosen_idx + len(split_token)
        rejected_split = rejected_idx + len(split_token)
        
        return {
            "prompt": example["chosen"][:chosen_split],
            "chosen": example["chosen"][chosen_split:].strip(),
            "rejected": example["rejected"][rejected_split:].strip()
        }
        # strip() will automatically delete any space, \t, \n at the end ir beginning of the sequence.


    pre_ds = pre_ds.map(split_prompt_response, remove_columns = pre_ds.column_names)


dpo_dataset = build_preference_dataset()

In [ ]:
dpo_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT_DIR)

if dpo_tokenizer.pad_token is None: 
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

policy_model = AutoModelForCausalLM.from_pretrained(
    SFT_OUTPUT_DIR,
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map = "auto",
)

dpo_lora_config = LoraConfig(
    r = LORA_R, 
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
    )


dpo_args = DPOConfig(
    output_dir = DPO_OUTPUT_DIR,
    beta = DPO_BETA, 
    num_train_epochs = DPO_EPOCHS,
    per_device_train_batch_size = DPO_BATCH_SIZE,
    gradient_accumulation_steps= DPO_GRAD_ACCUM,
    learning_rate = DPO_LR,
    max_length=MAX_SEQ_LEN,
    max_prompt_length=MAX_PROMPT_LEN,
    logging_steps=10,        
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model= policy_model,
    ref_model= None,               # LoRA + disabled adapters recovers pi_ref
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=dpo_tokenizer,
    peft_config=dpo_lora_config,
)

dpo_trainer.train()
dpo_trainer.save_model(DPO_OUTPUT_DIR)
dpo_tokenizer.save_pretrained(DPO_OUTPUT_DIR)
print(f"DPO-Tuned policy model save to {DPO_OUTPUT_DIR}")

In [ ]:
EVAL_PROMPTS = [
    "Explain what a black hole is to a 10 year old.",
    "Write a short, polite email declining a meeting invitation.",
    "What are three tips for staying focused while studying?",
    "Summarize the plot of Romeo and Juliet in two sentences.",
]


def generate(model, tokenizer, prompt, max_new_tokens = 128):
    text = f"### Instruction: \n {prompt}\n\n### Response:\n"
    inputs = tokenizer(text, return_tensors = "pt").to(model.device) 

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = True,
            top_p = 0.9, 
            temperature = 0.7,
            pad_token_id = tokenizer.eos_token_id,
        )


    full = tokenizer.decode(output[0], skip_special_tokens = True)
    return full[len(text):].strip() # chop off the input template and return only the ai generated text

# evaluating 
def sequence_logprob(model, tokenizer, prompt, response):
    
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
    full_ids = tokenizer(prompt + response, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        logits = model(full_ids).logits

    response_start = prompt_ids.shape[1]
    log_probs = torch.log_softmax(logits[:, :-1], dim=-1) # turining the logit into log-prob
    target_ids = full_ids[:, 1:]
    token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

    return token_log_probs[:, response_start - 1:].sum().item()


def implicit_reward(model, ref_model, tokenizer, prompt, response, beta=DPO_BETA):
    """r(x, y) = beta * log( pi_theta(y|x) / pi_ref(y|x) ) — Eq. from the paper's reparameterization."""
    policy_lp = sequence_logprob(model, tokenizer, prompt, response)
    ref_lp = sequence_logprob(ref_model, tokenizer, prompt, response)
    return beta * (policy_lp - ref_lp)


eval_tokenizer = AutoTokenizer.from_pretrained(SFT_OUTPUT_DIR)
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

sft_eval_model = AutoModelForCausalLM.from_pretrained(SFT_OUTPUT_DIR).eval()
dpo_eval_model = AutoModelForCausalLM.from_pretrained(DPO_OUTPUT_DIR).eval()

for prompt in EVAL_PROMPTS:
    print("=" * 80)
    print(f"PROMPT: {prompt}\n")

    text_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"

    sft_response = generate(sft_eval_model, eval_tokenizer, prompt)
    dpo_response = generate(dpo_eval_model, eval_tokenizer, prompt)


    sft_reward = implicit_reward(sft_eval_model, sft_eval_model, eval_tokenizer, text_prompt, sft_response)
    dpo_reward = implicit_reward(dpo_eval_model, sft_eval_model, eval_tokenizer, text_prompt, dpo_response)

    print(f"[SFT only]  (implicit reward={sft_reward:.3f})\n{sft_response}\n")
    print(f"[DPO-tuned] (implicit reward={dpo_reward:.3f})\n{dpo_response}\n")
    print(f"Reward delta (DPO - SFT): {dpo_reward - sft_reward:+.3f}")

print("=" * 80)